# Stage 8 — ICD-10-CM Code Packages

For each Stage 7 DiffDx diagnosis, build an **ordered ICD package**:

1. **Principal** — diagnosis → SNOMED → US ExtendedMap ICD-10-CM (`6011000124106`)
2. **Supporting** — related symptom-tree / retained SNOMED entities mapped to ICD  
   (relevance-filtered so they back *this* diagnosis, not the whole chart)

Map is authoritative (no free-form LLM code invention).  
Does **not** use current-stay ground-truth ICD.

**Input:** Stage 7 `differential_diagnoses.json` + `patient_records/` tree & retained  
**Output:** `data/stage_08_icd_coding/` + per-admission `icd_coding.json` / `.txt`


In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))
REPO = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR

from pipeline import (
    DIFF_DX_RESULTS_JSON,
    EXPORT_DIR,
    ICD_CODING_RESULTS_JSON,
    STAGE_08_DIR,
    print_pipeline_banner,
)
from snomed_ct import (
    build_icd10cm_map_index,
    build_snomed_index,
    export_stage08_to_patient_folders,
    find_snomed_root,
    run_stage08_icd_packages,
    write_json,
)

print_pipeline_banner()
STAGE_08_DIR.mkdir(parents=True, exist_ok=True)
print(f"Stage 7 in : {DIFF_DX_RESULTS_JSON}")
print(f"Stage 8 out: {STAGE_08_DIR}")
print(f"Export dir : {EXPORT_DIR}")
if not Path(DIFF_DX_RESULTS_JSON).exists():
    raise FileNotFoundError("Run stage_07_differential_diagnosis.ipynb first.")


Pipeline mode : FULL (15 patients)
LLM provider  : OpenRouter (qwen/qwen-2.5-7b-instruct, ZDR on)
Qwen pair     : Local equivalent: ollama pull qwen2.5:7b
Admissions/patient (min): 2
Data dir      : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data
Export dir    : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/patient_records
OpenRouter ZDR : enabled (provider.zdr=true on every request)
Stage 7 in : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/stage_07_differential_diagnosis/differential_diagnoses.json
Stage 8 out: /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/stage_08_icd_coding
Export dir : /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/patient_records


/Users/narenkhatwani/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
snomed_root = find_snomed_root(REPO / "data")
snomed_index = build_snomed_index(
    snomed_root=snomed_root,
    cache_path=REPO / "data" / "snomed_index" / "snomed_index.pkl",
    force_rebuild=False,
)
# First run builds ExtendedMap cache (~1–2 min); later runs load pickle
map_index = build_icd10cm_map_index(
    snomed_root=snomed_root,
    cache_path=REPO / "data" / "snomed_index" / "icd10cm_extended_map.pkl",
    force_rebuild=False,
    load_titles=True,
)
print(f"SNOMED concepts indexed: {len(snomed_index.active_concepts):,}")
print(f"ICD-10-CM mapped concepts: {len(map_index.concept_to_maps):,}")


Loading SNOMED index cache → /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/snomed_index/snomed_index.pkl
Loading ICD-10-CM ExtendedMap cache → /Users/narenkhatwani/GitHub/ai-agents-for-clinical-coding/data/snomed_index/icd10cm_extended_map.pkl
SNOMED concepts indexed: 382,170
ICD-10-CM mapped concepts: 127,974


In [3]:
stage07 = json.loads(Path(DIFF_DX_RESULTS_JSON).read_text(encoding="utf-8"))
print(f"Stage 7 admissions: {stage07.get('n_admissions')}")

payload = run_stage08_icd_packages(
    stage07,
    snomed_index,
    map_index,
    export_dir=EXPORT_DIR,
    use_embeddings=True,
)

n_princ = sum(
    (r.get("icd_enrichment") or {}).get("n_with_principal", 0)
    for r in payload["results"]
)
n_dx = sum(
    (r.get("icd_enrichment") or {}).get("n_diagnoses", 0)
    for r in payload["results"]
)
n_unmap = sum(
    (r.get("icd_enrichment") or {}).get("n_unmapped", 0)
    for r in payload["results"]
)
print(f"Diagnoses: {n_dx} | with principal ICD: {n_princ} | unmapped: {n_unmap}")

out = write_json(Path(ICD_CODING_RESULTS_JSON), payload)
n = export_stage08_to_patient_folders(payload, EXPORT_DIR)
print(f"Saved → {out}")
print(f"Per-admission files: {n} × icd_coding.json / .txt")


Stage 7 admissions: 15


/Users/narenkhatwani/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (local MiniLM)...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (local MiniLM)...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (local MiniLM)...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (local MiniLM)...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (local MiniLM)...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (local MiniLM)...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (local MiniLM)...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (local MiniLM)...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (local MiniLM)...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (local MiniLM)...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (local MiniLM)...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (local MiniLM)...
Loading embeddin

In [4]:
# Preview first admission with packages
for row in payload["results"]:
    diffs = row.get("differential") or []
    if not diffs:
        continue
    print(f"Patient {row.get('patient_id')} HADM {row.get('hadm_id')}")
    print(f"Most likely: {row.get('most_likely')}")
    for d in diffs[:3]:
        print(
            f"\n#{d.get('rank')} [{d.get('score')}] {d.get('diagnosis')} "
            f"| primary={d.get('icd10_primary')} | status={d.get('icd_status')}"
        )
        for c in (d.get("icd10_package") or [])[:6]:
            print(
                f"  {c.get('package_order')}. [{c.get('role')}] "
                f"{c.get('code')} — {c.get('title') or '(no title)'}"
            )
    break


Patient 10361982 HADM 24286431
Most likely: Postoperative hemorrhage

#1 [85.0] Postoperative hemorrhage | primary=N93.9 | status=mapped
  1. [principal] N93.9 — Abnormal uterine and vaginal bleeding, unspecified
  2. [supporting] N93.0 — Postcoital and contact bleeding
  3. [supporting] N92.0 — Excessive and frequent menstruation with regular cycle

#2 [70.0] Uterine atony | primary=O62.2 | status=mapped
  1. [principal] O62.2 — Other uterine inertia
  2. [supporting] N93.9 — Abnormal uterine and vaginal bleeding, unspecified

#3 [60.0] Infection | primary=B99.9 | status=mapped
  1. [principal] B99.9 — Unspecified infectious disease
  2. [principal] P39.9 — Infection specific to the perinatal period, unspecified
  3. [supporting] N93.9 — Abnormal uterine and vaginal bleeding, unspecified
